In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyabf
from scipy.signal import argrelextrema, find_peaks, savgol_filter

# =====================
# Configurable Parameters
# =====================
input_folder = "."  # Folder with ABF files
output_csv = "first_sweep_spike_summary.csv"
plot_folder = "first_sweep_plots"
threshold = 10       # mV spike threshold
prominence = 5      # Minimum prominence for spike detection
distance = 5        # Minimum distance between spikes (samples)
max_segment_length = 20  # Max samples before splitting segment
smooth = False        # Apply Savitzky-Golay smoothing

# Create plot folder if not exists
os.makedirs(plot_folder, exist_ok=True)

results = []

def split_long_segments(voltage, segment_indices, max_length=50):
    """
    Split long above-threshold segments at local minima.
    """
    new_segments = []
    for seg in segment_indices:
        start, end = seg
        if (end - start) > max_length:
            segment_data = voltage[start:end]
            minima_indices = argrelextrema(segment_data, np.less)[0]
            if len(minima_indices) > 0:
                split_points = [start + idx for idx in minima_indices]
                prev = start
                for sp in split_points:
                    new_segments.append((prev, sp))
                    prev = sp
                new_segments.append((prev, end))
            else:
                new_segments.append((start, end))
        else:
            new_segments.append((start, end))
    return new_segments

def detect_spike_peaks(time, voltage, threshold=10, prominence=10, distance=10, max_segment_length=50, smooth=True):
    """
    Detect spikes using threshold crossing, local minima splitting, and prominence-based peak detection.
    """
    # Optional smoothing
    if smooth:
        voltage_proc = savgol_filter(voltage, window_length=11, polyorder=3)
    else:
        voltage_proc = voltage

    # Step 1: Find above-threshold segments
    above = voltage_proc > threshold
    segments = []
    start = None
    for i, val in enumerate(above):
        if val and start is None:
            start = i
        elif not val and start is not None:
            segments.append((start, i))
            start = None
    if start is not None:
        segments.append((start, len(voltage_proc)))

    # Step 2: Split long segments at local minima
    segments = split_long_segments(voltage_proc, segments, max_segment_length)

    # Step 3: Find peaks in each segment using prominence
    spike_peaks = []
    for s, e in segments:
        seg_data = voltage_proc[s:e]
        peaks, _ = find_peaks(seg_data, height=threshold, prominence=prominence, distance=distance)
        spike_peaks.extend(s + peaks)

    return spike_peaks, voltage_proc

# =====================
# Process all ABF files (first sweep only)
# =====================
for file in os.listdir(input_folder):
    if file.lower().endswith(".abf"):
        filepath = os.path.join(input_folder, file)
        abf = pyabf.ABF(filepath)

        # First sweep only
        abf.setSweep(0)
        time = abf.sweepX
        voltage = abf.sweepY
        current = abf.sweepC if hasattr(abf, "sweepC") else None

        spike_peaks, processed_voltage = detect_spike_peaks(time, voltage, threshold, prominence, distance, max_segment_length, smooth)
        spike_times = time[spike_peaks]
        spike_voltages = voltage[spike_peaks]

        spike_count = len(spike_peaks)
        rheobase = current[spike_peaks[0]] if spike_count > 0 and current is not None else np.nan
        last_spike_current = current[spike_peaks[-1]] if spike_count > 0 and current is not None else np.nan
        isi = np.diff(spike_times) * 1000 if spike_count > 1 else []
        mean_isi = np.mean(isi) if len(isi) > 0 else np.nan

        # Plot first sweep
        plt.figure(figsize=(12, 6))
        plt.plot(time, processed_voltage, label="Voltage (mV)")
        plt.axhline(threshold, color="gray", linestyle="--", label="Threshold")
        if spike_peaks:
            plt.scatter(spike_times, processed_voltage[spike_peaks], color="red", label="Spike Peaks")
        plt.xlabel("Time (s)")
        plt.ylabel("Voltage (mV)")
        plt.title(f"{file} - First Sweep Spikes: {spike_count}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(plot_folder, f"{os.path.splitext(file)[0]}_first_sweep.png"))
        plt.close()

        # Append results
        results.append({
            "FileName": file,
            "SpikeCount": spike_count,
            "SpikeTimes_s": ",".join([f"{t:.4f}" for t in spike_times]),
            "SpikeVoltages_mV": ",".join([f"{v:.2f}" for v in spike_voltages]),
            "Rheobase_pA": rheobase,
            "LastSpikeCurrent_pA": last_spike_current,
            "Mean_ISI_ms": mean_isi
        })

# Save summary CSV
pd.DataFrame(results).to_csv(output_csv, index=False)